# <font color = 'red'> DEPENDENCIAS

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.miscmodels.ordinal_model import OrderedModel
from sklearn.preprocessing import MinMaxScaler

import sys
import os

# Agregar la carpeta calibration_code al path
sys.path.append(os.path.abspath("../../calibration_code"))

# Ahora puedes importar los módulos personalizados
from modelling_tools import (plot_histogram, plot_univariate_freq, assign_deciles, count_categories_by_decile, 
                             calculate_category_proportions, summarize_decile_analysis, summarize_grouped_deciles, group_deciles,
                             compute_odds_ratio)
from visualization_tools import plot_interactive_chart
from utils import g
from config import get_data_path, get_code_path
from data_cleaning import check_dataframe_quality

# <font color = 'red'> CARGA DE DATOS

In [2]:
df = pd.read_csv(get_data_path("bivariate_preprocessed_data.csv"))

In [3]:
res = check_dataframe_quality(df)

No missing values found.
No infinite values found.
No duplicate rows found.


# <font color = 'red'> ANÁLISIS

In [4]:
col = "Monthly_Balance"

## <font color = 'skyblue'> ANÁLISIS GENERAL

Las medianas tienen el orden esperado: Median Bad < Mediana Standard < Mediana Good.

Esto concuerda con la hipótesis de que a menor balance mayor proporción de malos.

In [5]:
fig_box = px.box(df, x="Credit_Mix", y=col, title=f"Distribution of {col} by Credit Score Category")
fig_box.show()

## <font color = 'skyblue'> ANÁLISIS POR DECILES

In [6]:
continuous_variable= col
decile_col_name = continuous_variable + '_Decile'
target_col_string = "Credit_Mix" # variable dependiente con nombres string
target_col = 'Credit_Score' # variable dependiente int (para modelos)

In [7]:
analysis_summary = summarize_decile_analysis(df, continuous_variable, decile_col_name, target_col_string)

# Obtener los resultados
df_deciles = analysis_summary["df_deciles"]  # DataFrame con los deciles asignados
deciles_summary = analysis_summary["decile_summary"]  # Resumen de deciles con conteos y proporciones
display(deciles_summary)
res = check_dataframe_quality(df_deciles)

,Decile_Min,Decile_Max,Decile_Count,Decile_Proportion,count_Bad,count_Good,count_Standard,prop_Bad,prop_Good,prop_Standard
Monthly_Balance_Decile,,,,,,,,,,
0,0.007760,218.511483,10000,0.1,4416,1861,3723,0.4416,0.1861,0.3723
1,218.514250,257.408384,10000,0.1,4547,1488,3965,0.4547,0.1488,0.3965
2,257.413365,282.560487,10000,0.1,4130,1404,4466,0.4130,0.1404,0.4466
3,282.561381,307.435011,10000,0.1,3319,1864,4817,0.3319,0.1864,0.4817
4,307.438910,337.267294,10000,0.1,2309,2497,5194,0.2309,0.2497,0.5194
5,337.274675,376.992495,10000,0.1,1795,3079,5126,0.1795,0.3079,0.5126
6,376.995288,433.451998,10000,0.1,1550,3552,4898,0.1550,0.3552,0.4898
7,433.454369,526.657547,10000,0.1,1218,4017,4765,0.1218,0.4017,0.4765
8,526.663884,710.553350,10000,0.1,478,4298,5224,0.0478,0.4298,0.5224


No missing values found.
No infinite values found.
No duplicate rows found.


In [8]:
df[(df[continuous_variable] >= 337.274675) & (df[continuous_variable] <= 376.992495) & (df['Credit_Score'] == 0)].shape

(1795, 92)

Proporción de Buenos: se observa una relación positiva, a mayor balance mensual mayor la proporción de buenos deudores.

Proporción de standard: la realación entre el balance mensual y la proporción de deudores Standard no es clara.

Proporción de malos: según lo esperado, la proporción de malos y el balance mensual se relacionan negativamente.

In [9]:
chart_types = {
    "prop_Good":"line",
    "prop_Standard":"line",
    "prop_Bad": "line",  
    "Decile_Count": "bar"    
}

fig = plot_interactive_chart(
    df=deciles_summary,  
    y_columns=["prop_Bad", "prop_Good", "prop_Standard", "Decile_Count"],  
    x_column="Decile_Max",  
    chart_types=chart_types, 
    title=f"Proportion of Credit Score Categories by {continuous_variable}",
    x_title="Decile",
    y_title="Decile Count",  
    y2_title="Proportion",   
    secondary_y=["prop_Bad", "prop_Good", "prop_Standard"],  
    width=900,
    height=500,
    custom_colors={"prop_Bad": "red", "prop_Good":"lightgreen",
    "prop_Standard":"brown", "Decile_Count": "gray"}  
)

fig.show()

Dado el comportamiento monótono por decil, no se proponen agrupaciones de estos.


## <font color = 'skyblue'> REGRESIONES BIVARIADAS

Como Credit_Score tiene tres categorías (Bad, Standard, Good) se pueden usar dos enfoques 
de regresión categórica: 

- Regresión Logística Multinomial → No asume orden en las categorías (como si fueran colores: rojo, azul, verde).
- Regresión Logística Ordinal → Asume que hay un orden en las categorías (Bad < Standard < Good).

Dado que hay un orden entre las categorías se utiliza Regresión Logística Ordinal:

<font color = 'gold'> Sin Agrupaciones

In [10]:
df_ = df_deciles.copy()
x_variable = continuous_variable
res = check_dataframe_quality(df_)

No missing values found.
No infinite values found.
No duplicate rows found.


In [11]:
# para no generar problemas numéricos debe escalarse esta variable.
# se opta por normalizar la variable:

g(df_[[x_variable]].describe()).transpose()

,count,mean,std,min,25%,50%,75%,max
Monthly_Balance,"100,000.00",403.49,214.42,0.01,270.32,337.27,471.93,"1,602.04"


Todos los coeficientes son significativos.

Monthly_Balance_Scaled 6.0847: el coeficiente es positivo: por cada dólar adicional en el balance mensual, la probabilidad de estar en una categoría superior de Credit_Score aumenta.

Threshold 0/1 0.2342: Umbral que separa las categorías Bad y Standard. Si la puntuación supera este umbral, es más probable que 
sea standard en lugar de bad.

Threshold 1/2 0.8026: Umbral que separa las categorías Standard y Good. Si la puntuación supera este umbral, es más probable que 
sea Good en lugar de Standard.

In [12]:
df_ = df_deciles.copy()
x_variable = continuous_variable

scaler = MinMaxScaler()
df_[continuous_variable + "_Scaled"] = scaler.fit_transform(df_[[x_variable]])

res = check_dataframe_quality(df_)

# Ajustar el modelo de regresión logística ordinal con la variable escalada
model_income = OrderedModel(df_[target_col], df_[continuous_variable + "_Scaled"], distr="logit")
result_income = model_income.fit(method='bfgs')

# Mostrar resumen del modelo
print(result_income.summary())

# Calcular e interpretar el Odds Ratio
res_odds = compute_odds_ratio(result_income, variable_name=continuous_variable + "_Scaled", description="Ingreso Anual Escalado")
print(res_odds["interpretation"])


No missing values found.
No infinite values found.
No duplicate rows found.
Optimization terminated successfully.
         Current function value: 0.983210
         Iterations: 13
         Function evaluations: 15
         Gradient evaluations: 15
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:                -98321.
Model:                   OrderedModel   AIC:                         1.966e+05
Method:            Maximum Likelihood   BIC:                         1.967e+05
Date:                Fri, 11 Apr 2025                                         
Time:                        11:10:35                                         
No. Observations:              100000                                         
Df Residuals:                   99997                                         
Df Model:                           1                                         
                             coef    std 

## <font color = 'skyblue'> EXPORTACIÓN DE DATOS CON VARIABLES ADICIONALES

In [ ]:
# df_deciles.to_csv("../../calibration_data/preprocessed_data.csv", index=False)
